# 03 — Synthetic YOLO dataset generator (+ augmentation)

Ports `dataset_generator.py`: composites transparent snack cutouts onto
background photos and writes YOLO-format `images/`, `labels/`, `data.yaml`.

New vs. the original script — data augmentation, since a fixed-pose,
fixed-lighting synthetic dataset generalizes poorly to the real spinning tray:

- random **rotation** (any angle, not just the photographed pose) + random
  horizontal flip
- **wider/configurable scale range** instead of a fixed 15–35% of frame width
- random placement across the **full frame**, including partial edge overlap
  (the tray edge can clip a snack as it spins into view)
- **lighting jitter** — brightness / contrast / saturation / color-temperature
  randomized per composite, to simulate different lighting hitting the tray
- optional slight blur/noise on the final composite, to simulate the tray's
  motion and camera sensor noise

`BACKGROUND_DIR` is the one line to change later once you have real photos of
the empty cardboard tray — right now it points at the existing generic stock
backgrounds as a placeholder (per your call to defer that for now).


In [ ]:
import os
import math
import random
import yaml
from PIL import Image, ImageEnhance, ImageFilter, UnidentifiedImageError
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches


## Config

In [ ]:
FG_FOLDER = "nutri_grain_strawberry_bar_transparent"
BACKGROUND_DIR = "../dev/programming/2026/Tufts-ai-final/background"  # PLACEHOLDER — swap for real cardboard-tray photos later
NUM_IMAGES = 300

# Augmentation ranges (tune these against what you observe on the real tray)
SCALE_RANGE = (0.12, 0.45)          # object width as a fraction of background width
ROTATION_RANGE = (0, 360)           # degrees
FLIP_PROB = 0.5
ALLOW_EDGE_OVERLAP = True           # if True, object may be partially cropped by the frame edge
EDGE_OVERLAP_FRACTION = 0.25        # max fraction of the object allowed to hang off-frame

BRIGHTNESS_RANGE = (0.7, 1.3)
CONTRAST_RANGE = (0.8, 1.2)
SATURATION_RANGE = (0.7, 1.3)

BLUR_PROB = 0.15
BLUR_RADIUS_RANGE = (0.5, 1.5)
NOISE_PROB = 0.25
NOISE_SIGMA_RANGE = (2, 8)  # in 0-255 pixel value units


## Augmentation helpers

In [ ]:
def augment_foreground_pose(fg_cropped):
    """Rotate + flip the cutout, then re-tighten the bounding box against the new alpha mask."""
    if random.random() < FLIP_PROB:
        fg_cropped = fg_cropped.transpose(Image.FLIP_LEFT_RIGHT)

    angle = random.uniform(*ROTATION_RANGE)
    rotated = fg_cropped.rotate(angle, resample=Image.BICUBIC, expand=True)

    tight_bbox = rotated.getbbox()
    if not tight_bbox:
        return None
    return rotated.crop(tight_bbox)


def augment_lighting(fg_rgba):
    """Jitter brightness/contrast/saturation on the RGB channels only, alpha untouched."""
    rgb = fg_rgba.convert("RGB")
    alpha = fg_rgba.getchannel("A")

    rgb = ImageEnhance.Brightness(rgb).enhance(random.uniform(*BRIGHTNESS_RANGE))
    rgb = ImageEnhance.Contrast(rgb).enhance(random.uniform(*CONTRAST_RANGE))
    rgb = ImageEnhance.Color(rgb).enhance(random.uniform(*SATURATION_RANGE))

    out = rgb.convert("RGBA")
    out.putalpha(alpha)
    return out


def augment_composite(bg_img):
    """Simulate camera/motion effects on the whole scene after compositing."""
    if random.random() < BLUR_PROB:
        bg_img = bg_img.filter(ImageFilter.GaussianBlur(radius=random.uniform(*BLUR_RADIUS_RANGE)))

    if random.random() < NOISE_PROB:
        arr = np.array(bg_img).astype(np.int16)
        sigma = random.uniform(*NOISE_SIGMA_RANGE)
        noise = np.random.normal(0, sigma, arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        bg_img = Image.fromarray(arr)

    return bg_img


## Core generator

Same overall algorithm as `dataset_generator.py` (pick random fg/bg, scale,
place, write YOLO label, build `data.yaml`), with the augmentation helpers
above inserted at the pose / lighting / final-composite stages.


In [ ]:
def generate_synthetic_yolo_dataset(fg_folder, bg_folder, output_folder=None, num_images=100):
    if output_folder is None:
        fg_basename = os.path.basename(os.path.normpath(fg_folder))
        output_folder = f"{fg_basename}_yolo_dataset"

    images_dir = os.path.join(output_folder, "images")
    labels_dir = os.path.join(output_folder, "labels")
    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)

    fg_files = [f for f in os.listdir(fg_folder) if f.lower().endswith(".png")]
    bg_files = [f for f in os.listdir(bg_folder) if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]

    if not fg_files:
        print(f"Error: No transparent PNGs found in '{fg_folder}'")
        return None
    if not bg_files:
        print(f"Error: No background images found in '{bg_folder}'")
        return None

    unique_classes = sorted(set(os.path.splitext(f)[0].replace("_transparent", "") for f in fg_files))
    class_to_id = {classname: idx for idx, classname in enumerate(unique_classes)}

    print(f"Detected classes: {list(class_to_id.keys())}")
    print(f"Generating {num_images} synthetic YOLO training images...")

    generated_count = 0
    attempts = 0
    max_attempts = num_images * 5  # augmentation adds more ways for a candidate to get discarded

    while generated_count < num_images and attempts < max_attempts:
        attempts += 1

        bg_name = random.choice(bg_files)
        fg_name = random.choice(fg_files)
        bg_path = os.path.join(bg_folder, bg_name)
        fg_path = os.path.join(fg_folder, fg_name)

        try:
            bg_img = Image.open(bg_path).convert("RGB")
            fg_img = Image.open(fg_path).convert("RGBA")
        except (UnidentifiedImageError, IOError):
            continue

        tight_bbox = fg_img.getbbox()
        if not tight_bbox:
            continue
        fg_cropped = fg_img.crop(tight_bbox)

        fg_posed = augment_foreground_pose(fg_cropped)
        if fg_posed is None:
            continue
        fg_lit = augment_lighting(fg_posed)

        label_name = os.path.splitext(os.path.basename(fg_path))[0].replace("_transparent", "")
        class_id = class_to_id[label_name]

        bg_w, bg_h = bg_img.size
        scale_factor = random.uniform(*SCALE_RANGE)
        new_w = int(bg_w * scale_factor)
        aspect_ratio = fg_lit.size[1] / fg_lit.size[0]
        new_h = int(new_w * aspect_ratio)
        if new_w <= 0 or new_h <= 0:
            continue

        fg_resized = fg_lit.resize((new_w, new_h), Image.Resampling.LANCZOS)

        if ALLOW_EDGE_OVERLAP:
            slack_x = int(new_w * EDGE_OVERLAP_FRACTION)
            slack_y = int(new_h * EDGE_OVERLAP_FRACTION)
            min_x, max_x = -slack_x, bg_w - new_w + slack_x
            min_y, max_y = -slack_y, bg_h - new_h + slack_y
        else:
            min_x, max_x = 0, bg_w - new_w
            min_y, max_y = 0, bg_h - new_h

        if max_x <= min_x or max_y <= min_y:
            continue

        paste_x = random.randint(min_x, max_x)
        paste_y = random.randint(min_y, max_y)

        bg_img.paste(fg_resized, (paste_x, paste_y), fg_resized)
        bg_img = augment_composite(bg_img)

        # Clip the label box to the visible frame (YOLO boxes must stay in [0, 1])
        vis_x0, vis_y0 = max(paste_x, 0), max(paste_y, 0)
        vis_x1, vis_y1 = min(paste_x + new_w, bg_w), min(paste_y + new_h, bg_h)
        vis_w, vis_h = vis_x1 - vis_x0, vis_y1 - vis_y0
        if vis_w <= 0 or vis_h <= 0:
            continue

        generated_count += 1
        base_filename = f"synthetic_{generated_count}"
        bg_img.save(os.path.join(images_dir, f"{base_filename}.jpg"), "JPEG")

        x_center = (vis_x0 + vis_w / 2.0) / bg_w
        y_center = (vis_y0 + vis_h / 2.0) / bg_h
        norm_w = vis_w / bg_w
        norm_h = vis_h / bg_h

        txt_path = os.path.join(labels_dir, f"{base_filename}.txt")
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")

        if generated_count % 10 == 0 or generated_count == num_images:
            print(f"  -> Generated {generated_count}/{num_images} files...")

    yaml_data = {
        "path": os.path.abspath(output_folder),
        "train": "images",
        "val": "images",  # image_rec_trainer.py splits this into train/val on first run
        "names": {idx: name for name, idx in class_to_id.items()},
    }
    yaml_path = os.path.join(output_folder, "data.yaml")
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

    print(f"\nDone! Dataset ready in: /{output_folder}  ({generated_count} images, {attempts} attempts)")
    return output_folder


## Intermediary test — generate 9 samples and visually check box alignment

Always run this before generating the full dataset. If the drawn boxes don't
tightly track the snack after rotation, the bbox re-tightening logic in
`augment_foreground_pose` needs a look before you waste GPU-hours training on
mislabeled boxes.


In [ ]:
def draw_yolo_grid(images_dir, labels_dir, names, n=9, cols=3):
    files = sorted(f for f in os.listdir(images_dir) if f.endswith(".jpg"))[:n]
    rows = math.ceil(len(files) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.atleast_1d(axes).flatten()

    for ax, fname in zip(axes, files):
        img = Image.open(os.path.join(images_dir, fname))
        w, h = img.size
        ax.imshow(img)

        label_path = os.path.join(labels_dir, os.path.splitext(fname)[0] + ".txt")
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    cls, xc, yc, bw, bh = map(float, line.split())
                    x0 = (xc - bw / 2) * w
                    y0 = (yc - bh / 2) * h
                    rect = patches.Rectangle((x0, y0), bw * w, bh * h, linewidth=2, edgecolor="lime", facecolor="none")
                    ax.add_patch(rect)
                    ax.text(x0, y0 - 4, names[int(cls)], color="lime", fontsize=8)
        ax.set_title(fname, fontsize=8)
        ax.axis("off")

    for ax in axes[len(files):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


TEST_OUTPUT = "_dataset_smoke_test"
test_result_dir = generate_synthetic_yolo_dataset(FG_FOLDER, BACKGROUND_DIR, output_folder=TEST_OUTPUT, num_images=9)

if test_result_dir:
    with open(os.path.join(test_result_dir, "data.yaml")) as f:
        test_yaml = yaml.safe_load(f)
    draw_yolo_grid(
        os.path.join(test_result_dir, "images"),
        os.path.join(test_result_dir, "labels"),
        test_yaml["names"],
    )


## Sanity check — image/label counts and class list line up

In [ ]:
def sanity_check_dataset(output_folder):
    images_dir = os.path.join(output_folder, "images")
    labels_dir = os.path.join(output_folder, "labels")
    n_images = len([f for f in os.listdir(images_dir) if f.endswith(".jpg")])
    n_labels = len([f for f in os.listdir(labels_dir) if f.endswith(".txt")])
    assert n_images == n_labels, f"image/label count mismatch: {n_images} images vs {n_labels} labels"

    with open(os.path.join(output_folder, "data.yaml")) as f:
        data_yaml = yaml.safe_load(f)
    print(f"OK: {n_images} images, {n_labels} labels, classes = {data_yaml['names']}")


sanity_check_dataset(TEST_OUTPUT)


## Full run

Only run once the 3x3 grid above looks right.

In [ ]:
RUN_FULL_GENERATION = False  # flip to True to generate the real dataset

if RUN_FULL_GENERATION:
    full_output_dir = generate_synthetic_yolo_dataset(FG_FOLDER, BACKGROUND_DIR, num_images=NUM_IMAGES)
    sanity_check_dataset(full_output_dir)
